# England Mortality & Deprivation Analytics

Author: Tanishk Nanasaheb Shinde

This walkthrough was added for the repository package. It reproduces descriptive findings from the supplied processed CSVs; it is not the original extraction or Tableau-authoring notebook. Run all cells from the repository root or the `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import json

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'scripts/analyze.py').is_file():
    raise FileNotFoundError('Open this notebook from the repository root or notebooks folder.')
spec = importlib.util.spec_from_file_location('analysis', ROOT / 'scripts/analyze.py')
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print('Project files located. Core analysis uses only the standard library.')

Project files located. Core analysis uses only the standard library.


## 1. Validate inputs and reproduce reports

The script validates analytical keys, unchanged mortality values after the deprivation join, complete confidence-interval bounds and latest indicator-summary values. Source-level checks requiring missing raw datasets are outside this reproduction.

In [2]:
summary = analysis.run(ROOT / 'data/processed', ROOT / 'reports')
print(json.dumps(summary['input_rows'], indent=2))
print(json.dumps(summary['validation'], indent=2))

{
  "mortality_trends": 110,
  "local_authority_comparison": 988,
  "deprivation_inequality": 988,
  "indicator_definitions": 5,
  "data_quality_summary": 15
}
{
  "national_duplicate_keys": 0,
  "local_duplicate_keys": 0,
  "deprivation_duplicate_keys": 0,
  "local_deprivation_keys_match": true,
  "local_deprivation_mortality_values_match": true,
  "reported_interval_bounds_valid": true,
  "latest_definitions_match": true
}


## 2. National changes

Relative change is `(latest / earliest - 1) * 100`. Rolling periods overlap, so the calculations describe changes rather than independent annual trend tests. Child mortality is per 100,000; the other indicators are per 1,000.

In [3]:
for row in summary['national']:
    print(f"{row['IndicatorName']}: {row['EarliestValue']:.2f} to {row['LatestValue']:.2f} {row['RateUnit']}; change {row['FullPeriodChangePct']:.1f}%")

Child mortality rate (1 to 17 years): 18.04 to 11.59 per 100,000; change -35.7%
Infant mortality rate: 5.36 to 4.17 per 1,000; change -22.2%
Stillbirth rate: 5.55 to 3.90 per 1,000; change -29.7%
Neonatal mortality rate: 3.61 to 3.07 per 1,000; change -15.1%
Post-neonatal mortality rate: 1.75 to 1.10 per 1,000; change -37.0%


## 3. Local coverage and England comparisons

Missing rates remain missing. The supplied Better/Similar/Worse categories should not be replaced by rank-based judgements.

In [4]:
for row in summary['local_coverage']:
    print(row['IndicatorName'], 'authorities:', row['Authorities'], 'available:', row['AvailableRates'], 'missing:', row['MissingRates'])
infant = next(r for r in summary['local_coverage'] if r['IndicatorID'] == '92196')
print({key: infant[key] for key in ['Similar', 'Better', 'Worse', 'Not compared']})

Child mortality rate (1 to 17 years) authorities: 132 available: 114 missing: 18
Infant mortality rate authorities: 296 available: 291 missing: 5
Stillbirth rate authorities: 296 available: 287 missing: 9
Neonatal mortality rate authorities: 132 available: 130 missing: 2
Post-neonatal mortality rate authorities: 132 available: 121 missing: 11
{'Similar': 238, 'Better': 28, 'Worse': 25, 'Not compared': 5}


## 4. Deprivation quintiles

These are unweighted authority means using supplied quintile labels. No grouping algorithm is invented. Historical deprivation data, incomplete coverage and ecological confounding limit interpretation.

In [5]:
for row in summary['infant_quintiles']:
    print(f"{row['Quintile']}: n={row['AvailableRates']}, mean={row['MeanRatePer1000']:.3f} per 1,000")
print(json.dumps(summary['quintile_coverage'], indent=2))
print(json.dumps(summary['quintile_gap'], indent=2))

Q1 Least deprived: n=47, mean=3.374 per 1,000
Q2: n=48, mean=3.574 per 1,000
Q3: n=48, mean=3.777 per 1,000
Q4: n=47, mean=4.318 per 1,000
Q5 Most deprived: n=51, mean=5.096 per 1,000
{
  "total_infant_authorities": 296,
  "assigned": 241,
  "unassigned": 55,
  "available_and_assigned": 241
}
{
  "Q5MinusQ1Per1000": 1.7224476399640816,
  "Q5OverQ1": 1.5105209480112336
}


## 5. Dashboard evidence

![Deprivation dashboard](../assets/dashboards/deprivation_mortality_inequality.png)

The dashboard PNG preserves the original view. Open `docs/DASHBOARD_GUIDE.md` for the complete gallery.

## 6. Known source discrepancy

The national trend image uses neonatal mortality, while the other national panels use infant mortality. The KPI image confidence interval also differs slightly from the CSV. The final presentation flags both issues.

In [6]:
print(summary['source_discrepancy'])
print('Reports are available in the reports folder.')

Supplied S2/dashboard KPI image shows CI 4.08–4.27. CSV CI rounds to 4.07–4.26.
Reports are available in the reports folder.


## Conclusion

Long-term declines coexist with recent increases. Local comparisons require attention to confidence intervals and missing values. The assigned deprivation quintiles show a gradient, but the area-level relationship does not establish a causal effect.